In [1]:
import sys
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import DataLoader

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists() and ROOT_DIR.parent.exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from song_recommender.paths import DATA_DIR, BASELINE_DIR, RESNET_MODELS_DIR, CONFIGS_DIR
from song_recommender.utils import load_config
from song_recommender.training import spec_baseline
from song_recommender.models import StemLateFusionResNet18, load_resnet_model, build_song_embeddings
from song_recommender.models.loader import load_tag_cluster_map, load_valid_tags
from song_recommender.data import StemSongDataset, TrackIndexer
from song_recommender.features.tag_features import add_tag_cluster_features
from song_recommender.evaluation import (
    topk_cosine, build_cluster_relevance_vector,
    precision_at_k, recall_at_k, average_precision_at_k,
    ndcg_at_k, jaccard_similarity, dominant_cluster_accuracy_at_k,
)

sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"device: {device}")

device: cuda


# Evaluation Metrics

We start this notebook by explaining each of the metrics that we will evaluate our models with. To start, we load the validation datset with the adjusted tag embeddings and clusters.

In [2]:
# Load config
config = load_config(CONFIGS_DIR / 'embeddings.yaml')

# Load the dataset files along with tag information
train_df = add_tag_cluster_features(pd.read_parquet(DATA_DIR / 'processed/train.parquet'), load_valid_tags(), load_tag_cluster_map())
val_df = add_tag_cluster_features(pd.read_parquet(DATA_DIR / 'processed/val.parquet'), load_valid_tags(), load_tag_cluster_map())

val_df.head(10)

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,deezer_artist,isrc,deezer_tags,deezer_year,tag_list,clean_tags,tag_set,tag_clusters,cluster_set,dominant_cluster
0,TRVEXNN128F930D87A,What Do You Want From Me,Pink Floyd,https://p.scdn.co/mp3-preview/3740f12813d62ac6...,0m44L9w4l6ClNcU48IZBAZ,"rock, classic_rock, progressive_rock, psychede...",NaN,1995,249400,0.483,...,Pink Floyd,GBN9X1100013,rock,2011,"[rock, classic_rock, progressive_rock, psyched...","[classic_rock, progressive_rock, psychedelic, ...","{progressive_rock, psychedelic, psychedelic_ro...","[16, 5, 5, 16]","{16, 5}",16.0
1,TRJBYTB128F42969D8,Whole Lotta Rosie,AC/DC,https://p.scdn.co/mp3-preview/eb3cf262b24943b0...,0V6mxSZnwc59wL8KJ8x6PM,"rock, classic_rock, hard_rock",Rock,1992,270200,0.257,...,AC/DC,AUAP07700011,"rock, hard rock, metal",1977,"[rock, classic_rock, hard_rock]","[classic_rock, hard_rock]","{hard_rock, classic_rock}","[16, 14]","{16, 14}",16.0
2,TRQLGEW128F1463DEA,Nobody's Fault but Mine,Led Zeppelin,https://p.scdn.co/mp3-preview/31bed1a977ba4e79...,2mYpbsbzCM3dqrM2XjgSjh,"rock, classic_rock, hard_rock, progressive_roc...",Rock,1976,376266,0.243,...,Led Zeppelin,USAT21500138,rock,2015,"[rock, classic_rock, hard_rock, progressive_ro...","[classic_rock, hard_rock, progressive_rock, br...","{progressive_rock, 70s, blues_rock, british, c...","[16, 14, 5, 20, 16, 16]","{16, 20, 5, 14}",16.0
3,TRRIYLO128F92FC3C9,Black Night,Deep Purple,https://p.scdn.co/mp3-preview/9cd18eb3354fa407...,06mRYYyrqHsVmGwLUbhb5f,"rock, classic_rock, hard_rock, progressive_roc...",NaN,1992,369173,0.404,...,Deep Purple,USWB19903559,pop,2000,"[rock, classic_rock, hard_rock, progressive_ro...","[classic_rock, hard_rock, progressive_rock, he...","{progressive_rock, 70s, classic_rock, hard_roc...","[16, 14, 5, 2, 16]","{16, 2, 5, 14}",16.0
4,TRXYUEY128F931CA3D,Set the Controls for the Heart of the Sun,Pink Floyd,https://p.scdn.co/mp3-preview/17ec5e555acf180e...,02lP5ANdHyw8MHF5SVqfA2,"rock, classic_rock, experimental, progressive_...",NaN,2001,320186,0.294,...,Pink Floyd,GBN9Y1100014,rock,2011,"[rock, classic_rock, experimental, progressive...","[classic_rock, experimental, progressive_rock,...","{psychedelic_rock, psychedelic, experimental, ...","[16, 5, 5, 20, 5, 16, 16]","{16, 20, 5}",16.0
5,TRVZBQO128F4295215,Coming Back to Life,Pink Floyd,https://p.scdn.co/mp3-preview/685067cc504201e4...,01at7a2Q4Eesz8HnK9ezIh,"rock, classic_rock, progressive_rock, 90s, psy...",NaN,1994,379400,0.292,...,Pink Floyd,GBN9X1100019,rock,2011,"[rock, classic_rock, progressive_rock, 90s, ps...","[classic_rock, progressive_rock, 90s, psychede...","{psychedelic_rock, psychedelic, classic_rock, ...","[16, 5, 19, 5, 16]","{16, 19, 5}",16.0
6,TRDKUTY128F933AD84,Ballad of a Thin Man,Bob Dylan,https://p.scdn.co/mp3-preview/40d8e7cd7cc409c8...,05Kf0ZRtmKaQxAd1vUGcxa,"classic_rock, folk, singer_songwriter, blues, ...",NaN,1985,420333,0.328,...,Bob Dylan,USSM19922505,rock,1965,"[classic_rock, folk, singer_songwriter, blues,...","[classic_rock, folk, singer_songwriter, blues,...","{blues_rock, folk, classic_rock, blues, americ...","[16, 17, 17, 16, 19, 16, 16]","{16, 17, 19}",16.0
7,TREBATJ128F4259D4A,She Sells Sanctuary,The Cult,https://p.scdn.co/mp3-preview/a33c6bd168ab5e75...,05DOyU6iDBgnbtUqWA1jCX,"rock, classic_rock, hard_rock, 80s",NaN,1994,252533,0.555,...,The Cult,GBAZP9700076,alternative,2000,"[rock, classic_rock, hard_rock, 80s]","[classic_rock, hard_rock, 80s]","{hard_rock, 80s, classic_rock}","[16, 14, 20]","{16, 20, 14}",16.0
8,TRMVZIF128F930B286,Southern Man,Neil Young,https://p.scdn.co/mp3-preview/83573edfee682003...,23eyDHPHW41eCZYga2q499,"rock, classic_rock, folk, singer_songwriter, 70s",NaN,2009,331600,0.377,...,Neil Young,USRE10900211,rock,1970,"[rock, classic_rock, folk, singer_songwriter, ...","[classic_rock, folk, singer_songwriter, 70s]","{folk, singer_songwriter, 70s, classic_rock}","[16, 17, 17, 16]","{16, 17}",16.0
9,TRGUIXO128F9318BEA,Qu

## Relevance

Before defining each metric, we need to define what it means for a recommended track to be *relevant* to the query track. We will say that the recommended track is relevant to the query track if

1. the query and the recommendation have the same `dominant_cluster`, or
2. they have significant cluster overlap.

We say that the two tracks have *significant cluster overlap* if the [Jaccard similarity index](https://en.wikipedia.org/wiki/Jaccard_index) is greater than or equal to a chosen threshold, $\tau$. More precisely, if $Q$ is the **set** of clusters for the query and $R$ is the **set** of clusters for the recommended track, we say that the overlap is significant if 

$$J(Q, R) = \frac{\lvert Q\cap R \rvert}{\lvert Q\cup R \rvert} \ge \tau.$$

For this notebook, we chose $\tau = 0.25$.

**Remark:** recall that `dominant_cluster` is determined by the mode of `tag_clusters`. If `tag_clusters` is multimodal, then the first appearing cluster in the list of multimodal clusters is selected. So while repetition of clusters in `tag_clusters` is not represented in the Jaccard index, it is represented in `dominant_cluster`.

## Metric Functions

For each query, we assigned a relevance vector $\mathbf{r} = \left[r_1, r_2, \ldots, r_k\right]^T$ using the tag embeddings and clusters, where $r_i = 1$ if the $i$-th recommended track is relevant to the query track and 0 otherwise for each $1 \le i \le k$.

We consider the following metrics for a query and its relevance vector:

1. Precision@k: number of relevant neighbors out of $k$
    - Measures recommendation quality, i.e., accuracy of relevant retrieval
    - Not rank aware
2. Recall@k: number of relevant neighbors out of total number of relevant items in dataset
    - Measures models ability to retrieve all relevant tracks
    - Will be very small if there are many "relevant" tracks, especially for small $k$
3. AP@k (Average Precision at k): Average of $r_i \cdot $ precision@i for $i = 1,\ldots, k$
    - Measures ranking quality
    - Penalizes relevant tracks appearing later in $k$-neighbors
4. nDCG@k ([Normalized Discounted Cumulative Gain](https://www.google.com/url?sa=i&source=web&rct=j&url=https://en.wikipedia.org/wiki/Discounted_cumulative_gain) at k): DCG@k over IDCG@k (Ideal DCG)
    * DCG@k is the sum of relevance results, discounted logarithmically by their position in $k$-neighbors:

    $$\text{DCG}@k = \sum_{i = 1}^k \frac{r_i}{\log_2(i+1)}$$
    * IDCG@k is the maximum possible DCG if $k$-neighbors were sorted by relevance, i.e., if $0 \le t \le k$ and $t$ of the $k$-neighbors are relevant, then $r_i > 0$ for all $i \le t$ and $$\text{IDCG}@k = \sum_{i = 1}^t \frac{r_i}{\log_2(i+1)}$$
    - Measures ranking quality
    - Rewards highly relevant tracks appearing earlier in $k$-neigbors
    - Grades relevance, e.g., highly relevant vs. partially relevant
    - Not as useful if relevance has a binary grading
    - Normalized for comparision across queries
5. Tag Jaccard Similarity: Average of tag Jaccard similarity for query against each $k$-neighbor
    - Measures if $k$-neighbors share tags
    - Not using relevancy vectors
    - Not used to compute relevancy
6. Cluster Jaccard Similarity: Average of cluster Jaccard similarity for query against each $k$-neighbor
    - Measures if $k$-neighbors share clusters
    - Not using relevancy vectors
    - Used to compute relevancy
7. Dominant Cluster Accuracy@k: number of neighbors whose dominant cluster matches query out of $k$

In [3]:
def evaluate_embeddings(
    query_emb, db_emb, query_df, db_df,
    k_list=(1, 5, 10, 20), overlap_threshold=0.25, self_retrieval=False,
):
    """
    Evaluate embedding retrieval quality using tag-cluster metrics.

    Args:
        query_emb:        (N_q, D) normalized embeddings for queries
        db_emb:           (N_db, D) normalized embeddings for database
        query_df/db_df:   DataFrames with cluster_set, dominant_cluster, tag_set columns
        k_list:           k values to evaluate
        overlap_threshold: Jaccard threshold for cluster overlap relevance
        self_retrieval:   if True, exclude index i from db (when query_emb is db_emb)

    Returns:
        dict {k: {metric_name: float}}
    """
    db_cluster_sets = db_df["cluster_set"].values
    db_dom_clusters = db_df["dominant_cluster"].values
    db_tag_sets     = db_df["tag_set"].values
    q_cluster_sets  = query_df["cluster_set"].values
    q_dom_clusters  = query_df["dominant_cluster"].values
    q_tag_sets      = query_df["tag_set"].values

    results = {}
    for k in k_list:
        prec, rec, ap, ndcg = [], [], [], []
        tag_j, clus_j, dom_acc = [], [], []

        for i in range(len(query_emb)):
            exclude = i if self_retrieval else None
            idx, _ = topk_cosine(db_emb, query_emb[i], k, exclude_index=exclude)

            relevance = build_cluster_relevance_vector(
                q_cluster_sets[i], db_cluster_sets[idx],
                q_dom_clusters[i], db_dom_clusters[idx],
                overlap_threshold,
            )
            full_rel = build_cluster_relevance_vector(
                q_cluster_sets[i], db_cluster_sets,
                q_dom_clusters[i], db_dom_clusters,
                overlap_threshold,
            )
            total_relevant = max(0, int(np.sum(full_rel)) - (1 if self_retrieval else 0))

            prec.append(precision_at_k(relevance))
            rec.append(recall_at_k(relevance, total_relevant))
            ap.append(average_precision_at_k(relevance))
            ndcg.append(ndcg_at_k(relevance))
            dom_acc.append(dominant_cluster_accuracy_at_k(q_dom_clusters[i], db_dom_clusters[idx]))
            tag_j.append(np.mean([jaccard_similarity(q_tag_sets[i], t) for t in db_tag_sets[idx]]))
            clus_j.append(np.mean([jaccard_similarity(q_cluster_sets[i], c) for c in db_cluster_sets[idx]]))

        results[k] = {
            "precision@k":            np.mean(prec),
            "recall@k":               np.mean(rec),
            "map@k":                  np.mean(ap),
            "ndcg@k":                 np.mean(ndcg),
            "tag_jaccard":            np.mean(tag_j),
            "cluster_jaccard":        np.mean(clus_j),
            "dominant_cluster_acc@k": np.mean(dom_acc),
        }
    return results

# Model Loading

In [4]:
# Baseline Model Information
baseline_emb = spec_baseline.build_embeddings(val_df, config)

# Load Validation Data
loader = DataLoader(StemSongDataset(val_df, image_size=224), batch_size=32, shuffle=False, num_workers=0)

# Resnet Notebook04 Information
model  = load_resnet_model( RESNET_MODELS_DIR / "04_Resnet18_contrastive_tags/checkpoint.pt", device)
out    = build_song_embeddings(model, loader, device)
resnet04_emb = out["song_embeddings"]

# Resnet Notebook05 Information
model  = load_resnet_model( RESNET_MODELS_DIR / "05_Resnet18_contrastive_tags_infonce/checkpoint.pt", device)
out    = build_song_embeddings(model, loader, device)
resnet05_emb = out["song_embeddings"]

# Future Resnet Models...

# Sanity Checking
print("baseline:", baseline_emb.shape)
print("resnet04:", resnet04_emb.shape)
print("resnet05:", resnet05_emb.shape)

embedding 1/1447
embedding 201/1447
embedding 401/1447
embedding 601/1447
embedding 801/1447
embedding 1001/1447
embedding 1201/1447
embedding 1401/1447
baseline: (1447, 5120)
resnet04: (1447, 64)
resnet05: (1447, 64)


In [5]:
k_list = [1, 5, 10, 20, 50]

models = {
    "Baseline":               baseline_emb,
    "ResNet04 (Contrastive)": resnet04_emb,
    "ResNet05 (InfoNCE)":     resnet05_emb,
}

all_results = {}
for name, emb in models.items():
    print(f"evaluating {name}...")
    all_results[name] = evaluate_embeddings(
        query_emb=emb, db_emb=emb,
        query_df=val_df, db_df=val_df,
        k_list=k_list, overlap_threshold=0.25, self_retrieval=True,
    )
print("done.")

evaluating Baseline...
evaluating ResNet04 (Contrastive)...
evaluating ResNet05 (InfoNCE)...
done.


In [7]:
k = 1
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.4112,0.0016,0.4112,0.4112,0.1274,0.2146,0.2308
ResNet04 (Contrastive),0.6185,0.0025,0.6185,0.6185,0.2007,0.3254,0.3649
ResNet05 (InfoNCE),0.6102,0.0025,0.6102,0.6102,0.2084,0.3338,0.3642


In [8]:
k = 5
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.3656,0.0074,0.5155,0.5960,0.1093,0.1932,0.1954
ResNet04 (Contrastive),0.6017,0.0124,0.7061,0.7708,0.2012,0.3227,0.3498
ResNet05 (InfoNCE),0.5906,0.0121,0.7001,0.7658,0.1988,0.3184,0.3504


In [6]:
k = 10
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.3534,0.0144,0.4889,0.6281,0.1023,0.1844,0.1845
ResNet04 (Contrastive),0.5954,0.0245,0.6801,0.7843,0.1951,0.3186,0.3457
ResNet05 (InfoNCE),0.5820,0.0238,0.6729,0.7793,0.1922,0.3112,0.3444


In [9]:
k = 20
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.3365,0.0274,0.4450,0.6432,0.0926,0.1730,0.1722
ResNet04 (Contrastive),0.5849,0.0480,0.6497,0.7922,0.1898,0.3101,0.3417
ResNet05 (InfoNCE),0.5718,0.0467,0.6404,0.7863,0.1833,0.3015,0.3367


In [10]:
k = 50
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.3114,0.0631,0.3880,0.6568,0.0824,0.1586,0.1562
ResNet04 (Contrastive),0.5652,0.1158,0.6140,0.8049,0.1780,0.2959,0.3260
ResNet05 (InfoNCE),0.5505,0.1120,0.6021,0.7974,0.1700,0.2864,0.3165


# Resnet model from Notebook 4

In [6]:
# Fresh-kernel reload cell: mirror the minimal model code needed for inference.
import json
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from torchvision.models import resnet18

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists() and ROOT_DIR.parent.exists():
    ROOT_DIR = ROOT_DIR.parent
SRC_DIR = ROOT_DIR / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

default_run_label = "04_Resnet18_contrastive_tags"
run_label = CFG.get("run_label", default_run_label) if "CFG" in globals() else default_run_label
data_dir = DATA_DIR if "DATA_DIR" in globals() else (ROOT_DIR / "data").resolve()

output_dir = (data_dir / "processed" / "model_runs" / run_label).resolve()
load_checkpoint_path = output_dir / "checkpoint.pt"
load_embeddings_path = output_dir / "embeddings.npz"
load_manifest_path = output_dir / "run_manifest.json"

loaded_val_df = pd.read_parquet(data_dir / "processed" / "val.parquet")
loaded_manifest = json.loads(load_manifest_path.read_text()) if load_manifest_path.exists() else None
if loaded_manifest is not None:
    print("loaded manifest:", load_manifest_path)

loaded_song_embeddings = None
loaded_mix_embeddings = None
loaded_stem_embeddings = None
loaded_track_ids = None

if load_embeddings_path.exists():
    loaded_npz = np.load(load_embeddings_path)
    loaded_song_embeddings = loaded_npz["song_embeddings"]
    loaded_mix_embeddings = loaded_npz["mix_embeddings"]
    loaded_stem_embeddings = loaded_npz["stem_embeddings"]
    loaded_track_ids = loaded_npz["spotify_id"].astype(str).tolist()
    print("loaded embeddings:", load_embeddings_path)
    print("song embeddings shape:", loaded_song_embeddings.shape)
    print("num track ids:", len(loaded_track_ids))
else:
    print("embeddings file not found:", load_embeddings_path)

if "ProjectionHead" not in globals():
    # Define the classes inline so this cell works without rerunning the training section.
    class ProjectionHead(nn.Module):
        def __init__(self, input_dim: int, projection_dim: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, input_dim),
                nn.ReLU(inplace=True),
                nn.Linear(input_dim, projection_dim),
            )

        def forward(self, x):
            return self.net(x)

if "StemLateFusionResNet18" not in globals():
    class StemLateFusionResNet18(nn.Module):
        def __init__(
            self,
            embedding_dim=64,
            projection_dim=64,
            pretrained=False,
            imagenet_input_norm=None,
            fusion_alpha_init=0.7,
            drum_alpha_init=0.3,
            num_stems=4,
            stem_dropout_prob=0.1,
            harmonic_indices=(0, 2, 3),
            drum_index=1,
        ):
            super().__init__()
            backbone = resnet18(weights=None)
            in_features = backbone.fc.in_features
            backbone.fc = nn.Linear(in_features, embedding_dim)

            self.encoder = backbone
            self.projection_head = ProjectionHead(embedding_dim, projection_dim)
            if imagenet_input_norm is None:
                imagenet_input_norm = pretrained
            self.imagenet_input_norm = bool(imagenet_input_norm)
            self.register_buffer(
                "imagenet_mean",
                torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1),
                persistent=False,
            )
            self.register_buffer(
                "imagenet_std",
                torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1),
                persistent=False,
            )

            self.fusion_alpha_logit = nn.Parameter(torch.tensor(fusion_alpha_init, dtype=torch.float32).logit())
            self.drum_alpha_logit = nn.Parameter(torch.tensor(drum_alpha_init, dtype=torch.float32).logit())
            self.num_stems = int(num_stems)
            self.harmonic_indices = tuple(int(idx) for idx in harmonic_indices)
            self.drum_index = int(drum_index)
            self.stem_logits = nn.Parameter(torch.zeros(self.num_stems, dtype=torch.float32))
            self.harmonic_logits = nn.Parameter(torch.zeros(len(self.harmonic_indices), dtype=torch.float32))
            self.register_buffer(
                "harmonic_index_tensor",
                torch.tensor(self.harmonic_indices, dtype=torch.long),
                persistent=False,
            )
            self.stem_dropout_prob = float(stem_dropout_prob)

        def _encode_inputs(self, x):
            if x.shape[1] == 1:
                x = x.repeat(1, 3, 1, 1)
            if self.imagenet_input_norm:
                x = (x - self.imagenet_mean) / self.imagenet_std
            return self.encoder(x)

        def forward(self, mix, stems):
            batch_size, num_stems, channels, height, width = stems.shape
            mix_embedding = self._encode_inputs(mix)
            stem_inputs = stems.reshape(batch_size * num_stems, channels, height, width)
            stem_embeddings = self._encode_inputs(stem_inputs).reshape(batch_size, num_stems, -1)
            stem_embeddings = F.normalize(stem_embeddings, dim=2, eps=1e-8)

            stem_weights = torch.softmax(self.stem_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
            stem_weights = stem_weights / stem_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)

            harmonic_stems = stem_embeddings[:, self.harmonic_index_tensor, :]
            harmonic_weights = torch.softmax(self.harmonic_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
            harmonic_weights = harmonic_weights * stem_weights[:, self.harmonic_index_tensor]
            harmonic_weights = torch.where(
                harmonic_weights.sum(dim=1, keepdim=True) == 0,
                torch.ones_like(harmonic_weights),
                harmonic_weights,
            )
            harmonic_weights = harmonic_weights / harmonic_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
            harmonic_embedding = torch.sum(harmonic_stems * harmonic_weights.unsqueeze(-1), dim=1)
            drum_embedding = stem_embeddings[:, self.drum_index, :] * stem_weights[:, self.drum_index].unsqueeze(-1)

            mix_embedding = F.normalize(mix_embedding, dim=1, eps=1e-8)
            harmonic_embedding = F.normalize(harmonic_embedding, dim=1, eps=1e-8)
            drum_embedding = F.normalize(drum_embedding, dim=1, eps=1e-8)

            alpha_h = torch.sigmoid(self.fusion_alpha_logit)
            alpha_d = torch.sigmoid(self.drum_alpha_logit)
            song_embedding = F.normalize(
                mix_embedding + alpha_h * harmonic_embedding + alpha_d * drum_embedding,
                dim=1,
                eps=1e-8,
            )

            harmonic_projection = self.projection_head(harmonic_embedding)
            return {
                "song_embedding": song_embedding,
                "mix_embedding": mix_embedding,
                "fused_stem_embedding": harmonic_embedding,
                "harmonic_embedding": harmonic_embedding,
                "drum_embedding": drum_embedding,
                "mix_projection": self.projection_head(mix_embedding),
                "harmonic_projection": harmonic_projection,
                "stem_projection": harmonic_projection,
            }

loaded_model = None
if load_checkpoint_path.exists():
    device = torch.device(CFG["device"] if "CFG" in globals() else ("cuda" if torch.cuda.is_available() else "cpu"))
    checkpoint = torch.load(load_checkpoint_path, map_location=device, weights_only=False)
    model_init_kwargs = checkpoint.get("model_init_kwargs", {}).copy()
    if not model_init_kwargs:
        checkpoint_cfg = checkpoint.get("cfg", {})
        model_init_kwargs = {
            "embedding_dim": int(checkpoint["embedding_dim"]),
            "projection_dim": int(checkpoint["projection_dim"]),
            "pretrained": False,
            "imagenet_input_norm": bool(checkpoint_cfg.get("pretrained", False)),
        }
    else:
        model_init_kwargs["pretrained"] = False
    loaded_model = StemLateFusionResNet18(**model_init_kwargs).to(device)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    print("loaded model:", load_checkpoint_path)
else:
    print("checkpoint not found:", load_checkpoint_path)

loaded manifest: C:\GitHub\dl-song-recommender\data\processed\model_runs\04_Resnet18_contrastive_tags\run_manifest.json
loaded embeddings: C:\GitHub\dl-song-recommender\data\processed\model_runs\04_Resnet18_contrastive_tags\embeddings.npz
song embeddings shape: (1690, 64)
num track ids: 1690
loaded model: C:\GitHub\dl-song-recommender\data\processed\model_runs\04_Resnet18_contrastive_tags\checkpoint.pt


In [8]:
def _topk_cosine_torch(embeddings, query_vec, k=10, exclude_idx=None):
    emb = torch.as_tensor(embeddings, dtype=torch.float32)
    q = torch.as_tensor(query_vec, dtype=torch.float32)
    sims = emb @ q
    if exclude_idx is not None:
        sims[exclude_idx] = float("-inf")
    k = min(int(k), int(emb.shape[0] - (1 if exclude_idx is not None else 0)))
    vals, idx = torch.topk(sims, k=k, largest=True, sorted=True)
    return idx.cpu().numpy(), vals.cpu().numpy()


def _recommend(df_in, emb, qid, k=10):
    qidx = int(df_in.index[df_in["spotify_id"] == qid][0])
    idx, vals = _topk_cosine_torch(emb, emb[qidx], k=k, exclude_idx=qidx)
    out = df_in.iloc[idx].copy()
    out["similarity"] = vals
    return out


if loaded_track_ids is None or loaded_song_embeddings is None:
    print("No loaded embeddings available for quick eval.")
else:
    loaded_embedding_df = loaded_val_df[["spotify_id", "artist", "name", "tags", "year"]].copy()
    # Reorder the saved rows to match the saved embedding matrix exactly.
    loaded_embedding_df = loaded_embedding_df.set_index("spotify_id").loc[loaded_track_ids].reset_index()

    query_id = CFG["query_spotify_id"] if "CFG" in globals() else loaded_embedding_df.iloc[0]["spotify_id"]
    if query_id not in set(loaded_embedding_df["spotify_id"]):
        query_id = loaded_embedding_df.iloc[0]["spotify_id"]
        print("Configured query id not found in loaded validation split; using first validation track instead:", query_id)

    recs_loaded = _recommend(loaded_embedding_df, loaded_song_embeddings, query_id, k=10)
    print("query id:", query_id)
    display(recs_loaded[["artist", "name", "spotify_id", "similarity"]])

KeyError: "None of [Index(['0nvIhBnscX9w7P2yrqxB6K', '0VmTwFFwHpyaaSmEIvGbpL',\n       '0gDz48UjL27r1svJvCLIG1', '0itHcaLfQmyKLajbbugfa6',\n       '0fzp2NFcfxnBevh0L4A9a1', '09BINxFxiDeywjfC39nRFi',\n       '11R0Wyq7eaxXsWbxfO4Ge7', '0GxyQIW2k4HTgcTpPbQapj',\n       '0v2VpAnwmGKaxBPikx20UJ', '0dBqYjpW2zuduO2EXB4FbQ',\n       ...\n       '0d3QbNg4OCII5uSvpucJ5N', '2Lgzb34x6izEIpVSLKlmVj',\n       '04t8JdbvEBUMUeShilCEyF', '4UMpyClJN1sM8CHgkKVOXD',\n       '004DCbObMToUiA5a17enh7', '0kIWHGFekI4N5j2KrFqKO8',\n       '2xhkgVYWR0GktOwpZeYj65', '0PEGMdCpjcWDPy5ebHL464',\n       '2wT5oRi5u10SiWGdkbEbIG', '6gFgahnOeNmjsifyzrlHbi'],\n      dtype='str', name='spotify_id', length=1690)] are in the [index]"

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

from song_recommender.models.loader import load_tag_cluster_map, load_valid_tags
from song_recommender.features.tag_features import add_tag_cluster_features
from song_recommender.evaluation import (
    topk_cosine, build_cluster_relevance_vector,
    precision_at_k, recall_at_k, average_precision_at_k,
    ndcg_at_k, jaccard_similarity, dominant_cluster_accuracy_at_k,
)

valid_tags = load_valid_tags()
tag_cluster_map = load_tag_cluster_map()
val_df = add_tag_cluster_features(loaded_val_df.copy(), valid_tags, tag_cluster_map)
display(val_df.head(3))

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,deezer_artist,isrc,deezer_tags,deezer_year,tag_list,clean_tags,tag_set,tag_clusters,cluster_set,dominant_cluster
0,TRVEXNN128F930D87A,What Do You Want From Me,Pink Floyd,https://p.scdn.co/mp3-preview/3740f12813d62ac6...,0m44L9w4l6ClNcU48IZBAZ,"rock, classic_rock, progressive_rock, psychede...",NaN,1995,249400,0.483,...,Pink Floyd,GBN9X1100013,rock,2011,"[rock, classic_rock, progressive_rock, psyched...","[classic_rock, progressive_rock, psychedelic, ...","{progressive_rock, psychedelic, psychedelic_ro...","[16, 5, 5, 16]","{16, 5}",16.0
1,TRJBYTB128F42969D8,Whole Lotta Rosie,AC/DC,https://p.scdn.co/mp3-preview/eb3cf262b24943b0...,0V6mxSZnwc59wL8KJ8x6PM,"rock, classic_rock, hard_rock",Rock,1992,270200,0.257,...,AC/DC,AUAP07700011,"rock, hard rock, metal",1977,"[rock, classic_rock, hard_rock]","[classic_rock, hard_rock]","{hard_rock, classic_rock}","[16, 14]","{16, 14}",16.0
2,TRQLGEW128F1463DEA,Nobody's Fault but Mine,Led Zeppelin,https://p.scdn.co/mp3-preview/31bed1a977ba4e79...,2mYpbsbzCM3dqrM2XjgSjh,"rock, classic_rock, hard_rock, progressive_roc...",Rock,1976,376266,0.243,...,Led Zeppelin,USAT21500138,rock,2015,"[rock, classic_rock, hard_rock, progressive_ro...","[classic_rock, hard_rock, progressive_rock, br...","{classic_rock, british, progressive_rock, 70s,...","[16, 14, 5, 20, 16, 16]","{16, 20, 5, 14}",16.0


In [4]:
def evaluate_embeddings(
    query_emb, db_emb, query_df, db_df,
    k_list=(5, 10, 20), overlap_threshold=0.25, self_retrieval=False,
):
    """
    Evaluate embedding retrieval quality against tag-cluster metrics.

    Args:
        query_emb:        (N_q, D) normalized embeddings for queries
        db_emb:           (N_db, D) normalized embeddings for database
        query_df/db_df:   DataFrames with cluster_set, dominant_cluster, tag_set columns
        k_list:           k values to evaluate
        overlap_threshold: Jaccard threshold for cluster overlap relevance
        self_retrieval:   if True, exclude index i from db (when query_emb is db_emb)

    Returns:
        dict {k: {metric_name: float}}
    """
    db_cluster_sets = db_df["cluster_set"].values
    db_dom_clusters = db_df["dominant_cluster"].values
    db_tag_sets     = db_df["tag_set"].values
    q_cluster_sets  = query_df["cluster_set"].values
    q_dom_clusters  = query_df["dominant_cluster"].values
    q_tag_sets      = query_df["tag_set"].values

    results = {}
    for k in k_list:
        prec, rec, ap, ndcg = [], [], [], []
        tag_j, clus_j, dom_acc = [], [], []

        for i in range(len(query_emb)):
            exclude = i if self_retrieval else None
            idx, _ = topk_cosine(db_emb, query_emb[i], k, exclude_index=exclude)

            relevance = build_cluster_relevance_vector(
                q_cluster_sets[i], db_cluster_sets[idx],
                q_dom_clusters[i], db_dom_clusters[idx],
                overlap_threshold,
            )
            full_rel = build_cluster_relevance_vector(
                q_cluster_sets[i], db_cluster_sets,
                q_dom_clusters[i], db_dom_clusters,
                overlap_threshold,
            )
            total_relevant = max(0, int(np.sum(full_rel)) - (1 if self_retrieval else 0))

            prec.append(precision_at_k(relevance))
            rec.append(recall_at_k(relevance, total_relevant))
            ap.append(average_precision_at_k(relevance))
            ndcg.append(ndcg_at_k(relevance))
            dom_acc.append(dominant_cluster_accuracy_at_k(q_dom_clusters[i], db_dom_clusters[idx]))
            tag_j.append(np.mean([jaccard_similarity(q_tag_sets[i], t) for t in db_tag_sets[idx]]))
            clus_j.append(np.mean([jaccard_similarity(q_cluster_sets[i], c) for c in db_cluster_sets[idx]]))

        results[k] = {
            "precision@k":            np.mean(prec),
            "recall@k":               np.mean(rec),
            "map@k":                  np.mean(ap),
            "ndcg@k":                 np.mean(ndcg),
            "tag_jaccard":            np.mean(tag_j),
            "cluster_jaccard":        np.mean(clus_j),
            "dominant_cluster_acc@k": np.mean(dom_acc),
        }
    return results

In [12]:
from torch.utils.data import DataLoader, Dataset
from song_recommender.data import TrackIndexer
from song_recommender.data.loader import load_png_resized

class StemSongDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_size: int = 224, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.image_size = image_size
        self.transform = transform
        self.indexer = TrackIndexer(self.df)

    def __len__(self) -> int:
        return len(self.df)

    def _load_spec(self, path) -> torch.Tensor:
        array = load_png_resized(str(path), image_size=self.image_size)
        return torch.as_tensor(array, dtype=torch.float32).unsqueeze(0)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        track_id = row["spotify_id"]
        # TrackIndexer returns the full mix first, then the stems in the checked project order.
        spec_paths = self.indexer.get_spec_png_paths(track_id)

        mix = self._load_spec(spec_paths[0])
        stems = torch.stack([self._load_spec(path) for path in spec_paths[1:]], dim=0)

        if self.transform is not None:
            mix, stems = self.transform(mix, stems)

        return {"track_id": track_id, "mix": mix, "stems": stems}
    
@torch.no_grad()
def build_song_embeddings(model, loader, device):
    # Encode the full validation split once, then reuse the cached matrix for retrieval and metrics.
    model.eval()
    track_ids = []
    song_embeddings = []
    mix_embeddings = []
    # Keep the old artifact name, but this is the harmonic branch used in the late fusion.
    stem_embeddings = []

    for batch in loader:
        mix = batch["mix"].to(device)
        stems = batch["stems"].to(device)
        out = model(mix, stems)

        track_ids.extend(batch["track_id"])
        song_embeddings.append(F.normalize(out["song_embedding"], dim=1, eps=1e-8).cpu())
        mix_embeddings.append(F.normalize(out["mix_embedding"], dim=1, eps=1e-8).cpu())
        stem_embeddings.append(F.normalize(out["fused_stem_embedding"], dim=1, eps=1e-8).cpu())

    return {
        "track_ids": track_ids,
        "song_embeddings": torch.cat(song_embeddings, dim=0).numpy(),
        "mix_embeddings": torch.cat(mix_embeddings, dim=0).numpy(),
        "stem_embeddings": torch.cat(stem_embeddings, dim=0).numpy(),
    }

In [13]:
image_size = loaded_manifest.get("image_size", 224) if loaded_manifest else 224
batch_size = 32

val_dataset = StemSongDataset(loaded_val_df, image_size=image_size, transform=None)
val_loader  = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

embedding_out   = build_song_embeddings(loaded_model, val_loader, device)
val_track_ids   = embedding_out["track_ids"]
val_song_emb    = embedding_out["song_embeddings"]
val_mix_emb     = embedding_out["mix_embeddings"]
val_stem_emb    = embedding_out["stem_embeddings"]

print("val song embeddings shape:", val_song_emb.shape)
print("num val track ids:", len(val_track_ids))

val song embeddings shape: (1447, 64)
num val track ids: 1447


In [15]:
results = evaluate_embeddings(
    query_emb=val_song_emb,
    db_emb=val_song_emb,
    query_df=val_df,
    db_df=val_df,
    k_list=[1, 5, 10, 20, 50],
    overlap_threshold=0.25,
    self_retrieval=True,
)
display(pd.DataFrame(results).T.rename_axis("k").round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
k,,,,,,,
1,0.6185,0.0025,0.6185,0.6185,0.2007,0.3254,0.3649
5,0.6017,0.0124,0.7061,0.7708,0.2012,0.3227,0.3498
10,0.5954,0.0245,0.6801,0.7843,0.1951,0.3186,0.3457
20,0.5849,0.0480,0.6497,0.7922,0.1898,0.3101,0.3417
50,0.5652,0.1158,0.6140,0.8049,0.1780,0.2959,0.3260
